# 08 Backend internals: adding your own

<a href="https://colab.research.google.com/github/bsc-wdc/qdislib/blob/main/notebooks/08_backend_internals.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a> <a href="https://mybinder.org/v2/gh/bsc-wdc/qdislib/main?labpath=notebooks/08_backend_internals.ipynb"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"></a>

Every execution target is a `BaseBackend`. The cutting code never branches on which one: it asks a backend object for counts. This notebook writes one, for a simulator
with a noise model. A real device works the same way.

In [1]:
# Installs Qdislib on Colab or Binder; does nothing where Qdislib 2 is installed.
try:
    import Qdislib
    assert Qdislib.__version__.startswith("2.")
except (ImportError, AssertionError):
    %pip install -q "qdislib[qiskit]>=2.0"

In [2]:
from qiskit import QuantumCircuit
import Qdislib as qd


def example_circuit():
    """Five qubits in a line, joined by four CZ gates."""
    qc = QuantumCircuit(5)
    for qubit, angle in enumerate([0.8, 0.5, 0.6, 0.4, 0.3]):
        qc.ry(angle, qubit)
    for qubit in range(4):
        qc.cz(qubit, qubit + 1)
    for qubit, angle in enumerate([0.2, 0.1, 0.7, 0.3, 0.9]):
        qc.rx(angle, qubit)
    return qc


circuit = example_circuit()
cut = ["CZ_7"]


exact = qd.analytical_solution(circuit, "ZZZZZ")

## One method: `evaluate_samples`

A subclass of `BaseBackend` needs one method: the final counts of a subcircuit, as `{bitstring: count}` with qubit 0 first. Everything else is derived from it, such as a
mid-circuit cut point, which is deferred onto an extra qubit read at the end.

`qd.sample_with_qiskit(dag, num_qubits, run)` writes it for a device that runs Qiskit circuits. It builds the measured circuit, calls your `run(circuit)` for `get_counts()`-style counts, and fixes the bit order.

This backend is a noisy Aer simulator, with a depolarizing error on two-qubit gates:

In [3]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error


class NoisyAer(qd.BaseBackend):
    """Aer with a depolarizing error on every two-qubit gate."""

    def evaluate_samples(self, dag, num_qubits, shots=1024, seed=None):
        noise = NoiseModel()
        noise.add_all_qubit_quantum_error(
            depolarizing_error(self.config.extra["error"], 2), ["cz", "cx"]
        )
        simulator = AerSimulator(noise_model=noise, seed_simulator=seed)
        return qd.sample_with_qiskit(
            dag, num_qubits, lambda qc: simulator.run(qc, shots=shots).result().get_counts()
        )


noisy = NoisyAer("NOISY_AER", qd.BackendConfig(num_qubits=20, extra={"error": 0.05}))

print(f"ideal {qd.gate_cutting(circuit, cut, shots=20_000, seed=1):+.4f}")
print(f"noisy {qd.gate_cutting(circuit, cut, shots=20_000, seed=1, backend=noisy):+.4f}")
print(f"exact {exact:+.4f}")

ideal +0.2354
noisy +0.2010
exact +0.2390


It already runs every cut type, sampling and three-step forms, with no other code:

In [4]:
print("wire   ", qd.wire_cutting(circuit, [("CZ_7", "CZ_8")], shots=20_000, seed=1, backend=noisy))
print("Hadamard", qd.gate_cutting_hadamard(circuit, cut, shots=20_000, seed=1, backend=noisy))
print("sampling", qd.gate_cutting_sampling(circuit, cut, shots=20_000, seed=1, backend=noisy, qubits=[0, 4]))

subs = qd.gate_cutting_subcircuits(circuit, cut)
print("3-step  ", subs.reconstruct([qd.sample_subcircuit(s, shots=20_000, seed=i, backend=noisy) for i, s in enumerate(subs)]))

wire    0.1941968675


Hadamard 0.20270333000000001
sampling {'00': 0.648541494375, '10': 0.13822358437499999, '01': 0.175776674375, '11': 0.03745824687500003}
3-step   0.20907247999999998


The same noise on the uncut circuit and on the cut one. On this circuit the cut value stays closer to the exact one, because a noisy CZ was replaced with local operations. That is this circuit and this noise model, not a general result: this model has no noise on measurements or single-qubit gates, and the cut pays in shots, not visible at this count.

In [5]:
def uncut(qc, error, shots=200_000, seed=1):
    noise = NoiseModel()
    noise.add_all_qubit_quantum_error(depolarizing_error(error, 2), ["cz"])
    measured = qc.copy()
    measured.measure_all()
    counts = AerSimulator(noise_model=noise, seed_simulator=seed).run(measured, shots=shots).result().get_counts()
    return sum((-1) ** bits.count("1") * c for bits, c in counts.items()) / shots


for error in (0.0, 0.01, 0.05):
    device = NoisyAer("NOISY_AER", qd.BackendConfig(num_qubits=20, extra={"error": error}))
    print(f"error {error:4}: uncut {uncut(circuit, error):+.4f}  "
          f"cut {qd.gate_cutting(circuit, cut, shots=200_000, seed=1, backend=device):+.4f}")

error  0.0: uncut +0.2376  cut +0.2382


error 0.01: uncut +0.2294  cut +0.2306


error 0.05: uncut +0.1919  cut +0.2037


Aer's own backend takes a noise model too, `qd.build_backend("AER_BACKEND", qd.BackendConfig(extra={"noise_model": noise}))`, and `NoiseModel.from_backend` works. Writing a backend is for what the provided ones cannot do.

## Registering it by name

`qd.register_backend(name, module, class_name)` makes a backend available by name, imported only when first used. It then needs a constructor that takes a config, and a module
that workers can import.

In [6]:
import pathlib
import sys
import tempfile

folder = tempfile.mkdtemp()
pathlib.Path(folder, "my_device.py").write_text("""
import Qdislib as qd
from qiskit_aer import AerSimulator


class MyDevice(qd.BaseBackend):
    def __init__(self, config):
        super().__init__("MY_DEVICE", config)

    def evaluate_samples(self, dag, num_qubits, shots=1024, seed=None):
        simulator = AerSimulator(seed_simulator=seed)
        return qd.sample_with_qiskit(
            dag, num_qubits, lambda qc: simulator.run(qc, shots=shots).result().get_counts()
        )
""")
sys.path.insert(0, folder)

qd.register_backend("MY_DEVICE", "my_device", "MyDevice")
print(qd.available_backends())
print(qd.gate_cutting(circuit, cut, shots=20_000, seed=1, backend="MY_DEVICE"))

['AER_BACKEND', 'QIBO_BACKEND', 'CUDAQ_BACKEND', 'PENNYLANE_BACKEND', 'QILISDK_BACKEND', 'IQM_BACKEND', 'BSC_QPU_BACKEND', 'IBM_BACKEND', 'MY_DEVICE']
0.23536028


## Optional methods

| Method or property | When |
|---|---|
| `evaluate_joint_counts` | the device reports mid-circuit outcomes, so no extra qubit is needed |
| `evaluate_joint_counts_many`, `packs` | the device runs several circuits in one job (the IBM backend's `pack=True`, notebook 13) |
| `uses_gpu` | the backend is explicitly on a GPU: under PyCOMPSs its circuits then run in tasks that ask for one GPU and one CPU core |
| `evaluate_expectation` | for `qd.expectation_value` and the deprecated 1.x paths |
| `transpile`, `_execute_samples`, `_execute_estimator` | for `QdBackEnd`'s queue |

`uses_gpu` must be true only when the backend really is on a GPU: under PyCOMPSs those tasks wait forever on a cluster without GPUs.

In [7]:
print("the backend above:", noisy.uses_gpu)


class MyGpuDevice(NoisyAer):
    @property
    def uses_gpu(self):
        return True


print("your own:         ", MyGpuDevice("GPU", qd.BackendConfig(num_qubits=20)).uses_gpu)

the backend above: False
your own:          True


## A pool of devices: `QdBackEnd`

`QdBackEnd` holds a pool of backends and routes each whole circuit. It keeps those with enough qubits, tries the shortest queue first, and falls back to the next if one fails. It is independent of cutting;
for cutting, choose a device per subcircuit in the three-step loop with `s.num_qubits` (notebook 02).

In [8]:
pool = qd.QdBackEnd(qpu_dict={"AER_BACKEND": {"num_qubits": 30}})

qc = QuantumCircuit(2)
qc.h(0)
qc.cz(0, 1)
qc.h(1)
qc.measure_all()

result = pool.sample_circuit(qc, shots=1024)
print(result.counts)
print(pool.get_expectation_value(qc, "ZZ", shots=1024).expectation_value)

{'01': 263, '11': 235, '00': 269, '10': 257}


0.033203125


Next: **09 Distributed execution and QPUs**.